# 04 - Model Comparison

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

In [2]:
# Load cleaned data
df = pd.read_csv("cleaned.csv", dtype={"CloseMonth": str})
months = sorted(df["CloseMonth"].unique())
df.shape

(60477, 74)

## Train/test split

In [3]:
# Test on the most recent month, train on the X months before it
X = 4
train = df[df["CloseMonth"].isin(months[-1 - X:-1])].copy()
test = df[df["CloseMonth"] == months[-1]].copy()
train.shape, test.shape

((38411, 74), (11788, 74))

In [4]:
# Normalize numeric features using the training set's mean and std
num_cols = ["LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeSquareFeet", "YearBuilt",
            "GarageSpaces", "ParkingTotal", "AssociationFee", "Latitude", "Longitude"]
mean, std = train[num_cols].mean(), train[num_cols].std()
train[num_cols] = (train[num_cols] - mean) / std
test[num_cols] = (test[num_cols] - mean) / std

## Models

In [7]:
features = [c for c in df.columns if c not in ["ClosePrice", "CloseMonth"]]

models = {
    "LinearRegression": LinearRegression(),
    "DecisionTree": DecisionTreeRegressor(),
    "RandomForest": RandomForestRegressor(n_jobs=-1),
}

rows = []
for name, model in models.items():
    model.fit(train[features], train["ClosePrice"])
    train_r2 = r2_score(train["ClosePrice"], model.predict(train[features]))
    test_r2 = r2_score(test["ClosePrice"], model.predict(test[features]))
    rows.append((name, train_r2, test_r2))

## Results

In [8]:
results = pd.DataFrame(rows, columns=["model", "train_R2", "test_R2"])
results

,model,train_R2,test_R2
0,LinearRegression,0.579540,0.595410
1,DecisionTree,0.998156,0.643106
2,RandomForest,0.972203,0.826921


## Model behavior

**Linear Regression**
- Strengths: fast, simple, and easy to interpret through its coefficients. Train and test R² are close, so it isn't overfitting.
- Weaknesses: It can only fit straight-line relationships. It is also sensitive to extreme prices which helped me realize I did not do a great job removing outliers.

**Decision Tree**
- Strengths: Captures non-linear effects and interactions without any feature engineering. Decision trees are also easy to visualize.
- Weaknesses: With no depth limit it memorizes the training data and generalizes poorly. Small changes in the data can produce a very different tree.

**Random Forest**
- Strengths: It has the best test R². Averaging many trees cancels out much of a single tree's overfitting.
- Weaknesses: slower to train, harder to interpret than a single tree or linear model, and still overfits somewhat.
